# Localization Training Analysis

This notebook analyzes `history.json` files produced by `localization.train.trainer`.

It supports:
- single-run analysis
- multi-run comparison
- smoothing
- best-epoch marking
- plotting all standard metrics
- plotting arbitrary metrics found in `history.json`
- saving figures

Typical history keys include:
- train_total_loss
- train_heat_loss
- train_size_loss
- val_total_loss
- val_heat_loss
- val_size_loss
- median_center_error_mm
- mean_center_error_mm
- p_at_thresh
- mean_iou

### Import

In [ ]:
from pathlib import Path
import json
import math
from typing import Any, Dict, List, Sequence, Optional

import numpy as np
import matplotlib.pyplot as plt

### Easy-Summary

In [ ]:
print("Single-run quick summary")
print("------------------------")
print("Best epoch (medCE):", best_epoch_for_metric(history, "median_center_error_mm", maximize=False))

for key, maximize in [
    ("val_total_loss", False),
    ("val_heat_loss", False),
    ("val_size_loss", False),
    ("median_center_error_mm", False),
    ("mean_center_error_mm", False),
    ("p_at_thresh", True),
    ("mean_iou", True),
]:
    if key in list_metric_keys(history):
        print(
            f"{key:24s} | best = {best_value(history, key, maximize=maximize):.6f} | "
            f"final = {final_value(history, key):.6f}"
        )

plot_standard_single_run(history, smooth=3)

### Functions

In [ ]:
HistoryDict = Dict[str, Any]


def load_history(path: str | Path) -> HistoryDict:
    path = Path(path)
    with open(path, "r") as f:
        return json.load(f)


def get_epoch_records(history: HistoryDict) -> List[Dict[str, Any]]:
    return list(history.get("epochs", []))


def get_epoch_numbers(history: HistoryDict) -> List[int]:
    records = get_epoch_records(history)
    return [int(rec.get("epoch", i + 1)) for i, rec in enumerate(records)]


def extract_series(history: HistoryDict, key: str, default: float = float("nan")) -> List[float]:
    vals = []
    for rec in get_epoch_records(history):
        v = rec.get(key, default)
        try:
            vals.append(float(v))
        except Exception:
            vals.append(float("nan"))
    return vals


def moving_average(values: Sequence[float], window: int = 1) -> List[float]:
    vals = [float(v) for v in values]
    if window <= 1 or len(vals) == 0:
        return vals

    out = []
    for i in range(len(vals)):
        start = max(0, i - window + 1)
        chunk = vals[start:i + 1]
        chunk = [v for v in chunk if np.isfinite(v)]
        out.append(float(np.mean(chunk)) if len(chunk) > 0 else float("nan"))
    return out


def best_epoch_for_metric(history: HistoryDict, metric_key: str, maximize: bool = False) -> Optional[int]:
    best_epoch = None
    best_value = None

    for rec in get_epoch_records(history):
        if metric_key not in rec:
            continue

        try:
            value = float(rec[metric_key])
        except Exception:
            continue

        if not np.isfinite(value):
            continue

        if best_value is None:
            best_value = value
            best_epoch = int(rec.get("epoch", 0))
            continue

        better = (value > best_value) if maximize else (value < best_value)
        if better:
            best_value = value
            best_epoch = int(rec.get("epoch", 0))

    return best_epoch


def final_value(history: HistoryDict, key: str) -> float:
    ys = extract_series(history, key)
    ys = [y for y in ys if np.isfinite(y)]
    return ys[-1] if len(ys) > 0 else float("nan")


def best_value(history: HistoryDict, key: str, maximize: bool = False) -> float:
    ys = extract_series(history, key)
    ys = [y for y in ys if np.isfinite(y)]
    if len(ys) == 0:
        return float("nan")
    return max(ys) if maximize else min(ys)


def list_metric_keys(history: HistoryDict) -> List[str]:
    keys = set()
    for rec in get_epoch_records(history):
        keys.update(rec.keys())
    keys.discard("epoch")
    return sorted(keys)


def split_metric_keys(history: HistoryDict):
    keys = list_metric_keys(history)

    train_keys = [k for k in keys if k.startswith("train_")]
    val_keys = [k for k in keys if k.startswith("val_")]
    error_keys = [k for k in keys if "error" in k.lower()]
    iou_keys = [k for k in keys if "iou" in k.lower()]
    prob_keys = [k for k in keys if "p_at" in k.lower() or "success" in k.lower()]
    loss_keys = [k for k in keys if "loss" in k.lower()]
    other_keys = [k for k in keys if k not in set(train_keys + val_keys + error_keys + iou_keys + prob_keys + loss_keys)]

    return {
        "train": train_keys,
        "val": val_keys,
        "error": error_keys,
        "iou": iou_keys,
        "prob": prob_keys,
        "loss": loss_keys,
        "other": other_keys,
    }

### Plot Functions

In [ ]:
def plot_single_run(
    history: HistoryDict,
    keys: Sequence[str],
    title: str,
    ylabel: str = "",
    smooth: int = 1,
    percent_keys: Optional[Sequence[str]] = None,
    best_metric_key: Optional[str] = "median_center_error_mm",
    maximize_best_metric: bool = False,
    figsize=(10, 6),
    logy: bool = False,
):
    percent_keys = set(percent_keys or [])
    xs = get_epoch_numbers(history)

    plt.figure(figsize=figsize)

    for key in keys:
        ys = extract_series(history, key)
        if key in percent_keys:
            ys = [100.0 * y if np.isfinite(y) else y for y in ys]
        ys = moving_average(ys, smooth)
        plt.plot(xs, ys, label=key)

    if best_metric_key is not None:
        be = best_epoch_for_metric(history, best_metric_key, maximize=maximize_best_metric)
        if be is not None:
            plt.axvline(be, linestyle="--", alpha=0.5, label=f"best epoch ({be})")

    plt.title(title)
    plt.xlabel("Epoch")
    plt.ylabel(ylabel)
    plt.grid(True, alpha=0.3)
    if logy:
        plt.yscale("log")
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_multi_run(
    histories: Sequence[HistoryDict],
    labels: Sequence[str],
    key: str,
    title: Optional[str] = None,
    ylabel: str = "",
    smooth: int = 1,
    as_percent: bool = False,
    best_metric_key: Optional[str] = None,
    maximize_best_metric: bool = False,
    figsize=(10, 6),
    logy: bool = False,
):
    plt.figure(figsize=figsize)

    for history, label in zip(histories, labels):
        xs = get_epoch_numbers(history)
        ys = extract_series(history, key)
        if as_percent:
            ys = [100.0 * y if np.isfinite(y) else y for y in ys]
        ys = moving_average(ys, smooth)

        plt.plot(xs, ys, label=label)

        if best_metric_key is not None:
            be = best_epoch_for_metric(history, best_metric_key, maximize=maximize_best_metric)
            if be is not None:
                plt.axvline(be, linestyle="--", alpha=0.25)

    plt.title(title or key)
    plt.xlabel("Epoch")
    plt.ylabel(ylabel or key)
    plt.grid(True, alpha=0.3)
    if logy:
        plt.yscale("log")
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_metric_grid(
    history: HistoryDict,
    keys: Sequence[str],
    ncols: int = 2,
    smooth: int = 1,
    percent_keys: Optional[Sequence[str]] = None,
    best_metric_key: Optional[str] = "median_center_error_mm",
    maximize_best_metric: bool = False,
    figsize_per_plot=(7, 4),
):
    percent_keys = set(percent_keys or [])
    xs = get_epoch_numbers(history)

    n = len(keys)
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=(figsize_per_plot[0] * ncols, figsize_per_plot[1] * nrows),
    )

    axes = np.array(axes).reshape(-1)

    for ax, key in zip(axes, keys):
        ys = extract_series(history, key)
        if key in percent_keys:
            ys = [100.0 * y if np.isfinite(y) else y for y in ys]
        ys = moving_average(ys, smooth)

        ax.plot(xs, ys, label=key)
        if best_metric_key is not None:
            be = best_epoch_for_metric(history, best_metric_key, maximize=maximize_best_metric)
            if be is not None:
                ax.axvline(be, linestyle="--", alpha=0.4)

        ax.set_title(key)
        ax.set_xlabel("Epoch")
        ax.set_ylabel(key)
        ax.grid(True, alpha=0.3)
        ax.legend()

    for ax in axes[len(keys):]:
        ax.axis("off")

    fig.tight_layout()
    plt.show()


def save_current_figure(path: str | Path, dpi: int = 150):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(path, dpi=dpi, bbox_inches="tight")

### Standard Plot Package

In [ ]:
def plot_standard_single_run(history: HistoryDict, smooth: int = 1):
    plot_single_run(
        history,
        keys=["train_total_loss", "val_total_loss"],
        title="Total Loss",
        ylabel="Loss",
        smooth=smooth,
    )

    plot_single_run(
        history,
        keys=["train_heat_loss", "val_heat_loss", "train_size_loss", "val_size_loss"],
        title="Heat / Size Loss",
        ylabel="Loss",
        smooth=smooth,
    )

    plot_single_run(
        history,
        keys=["median_center_error_mm", "mean_center_error_mm"],
        title="Center Error",
        ylabel="mm",
        smooth=smooth,
    )

    plot_single_run(
        history,
        keys=["p_at_thresh"],
        title="Success Rate",
        ylabel="Percent (%)",
        smooth=smooth,
        percent_keys=["p_at_thresh"],
    )

    plot_single_run(
        history,
        keys=["mean_iou"],
        title="Mean IoU",
        ylabel="IoU",
        smooth=smooth,
    )

### Standard Analysis

In [ ]:
plot_standard_single_run(history, smooth=1)

In [ ]:
plot_standard_single_run(history, smooth=5)

### Metrics and Grids

In [ ]:
metric_groups = split_metric_keys(history)

print("Loss keys:", metric_groups["loss"])
print("Error keys:", metric_groups["error"])
print("IoU keys:", metric_groups["iou"])
print("Prob keys:", metric_groups["prob"])
print("Other keys:", metric_groups["other"])

In [ ]:
all_keys = [k for k in list_metric_keys(history) if k != "epoch"]

plot_metric_grid(
    history,
    keys=all_keys,
    ncols=2,
    smooth=3,
    percent_keys=["p_at_thresh"],
)

### Special Metrics

In [ ]:
plot_single_run(
    history,
    keys=["train_size_loss", "val_size_loss"],
    title="Only Size Loss",
    ylabel="Loss",
    smooth=1,
)

In [ ]:
plot_single_run(
    history,
    keys=["median_center_error_mm", "mean_center_error_mm"],
    title="Center Error Detailed",
    ylabel="mm",
    smooth=3,
)

In [ ]:
plot_single_run(
    history,
    keys=["val_total_loss", "median_center_error_mm"],
    title="Mixed Metrics View",
    ylabel="Value",
    smooth=1,
)

### Log-Scale Plot

In [ ]:
plot_single_run(
    history,
    keys=["train_heat_loss", "val_heat_loss", "train_size_loss", "val_size_loss"],
    title="Heat / Size Loss (log scale)",
    ylabel="Loss",
    smooth=1,
    logy=True,
)

### Best Epoch Summary

In [ ]:
best_epoch = best_epoch_for_metric(history, "median_center_error_mm", maximize=False)
print("Best epoch by median_center_error_mm:", best_epoch)

for key, maximize in [
    ("median_center_error_mm", False),
    ("mean_center_error_mm", False),
    ("p_at_thresh", True),
    ("mean_iou", True),
    ("val_total_loss", False),
    ("val_heat_loss", False),
    ("val_size_loss", False),
]:
    if key in list_metric_keys(history):
        print(
            f"{key:24s} | best = {best_value(history, key, maximize=maximize):.6f} | "
            f"final = {final_value(history, key):.6f}"
        )

### Multiple Run Comparison

In [ ]:
RUNS = {
    "unet": Path("../outputs/unet_run/history.json"),
    "cnn": Path("../outputs/cnn_run/history.json"),
    "resnet": Path("../outputs/resnet_run/history.json"),
}

histories = [load_history(p) for p in RUNS.values()]
labels = list(RUNS.keys())

print(labels)

#### Total Loss Comparison

In [ ]:
plot_multi_run(
    histories,
    labels,
    key="val_total_loss",
    title="Validation Total Loss Comparison",
    ylabel="Loss",
    smooth=3,
    best_metric_key="median_center_error_mm",
    maximize_best_metric=False,
)

#### Center Error Comparison

In [ ]:
plot_multi_run(
    histories,
    labels,
    key="median_center_error_mm",
    title="Median Center Error Comparison",
    ylabel="mm",
    smooth=3,
    best_metric_key="median_center_error_mm",
    maximize_best_metric=False,
)

#### P@thresh Comparison

In [ ]:
plot_multi_run(
    histories,
    labels,
    key="p_at_thresh",
    title="Success Rate Comparison",
    ylabel="Percent (%)",
    smooth=3,
    as_percent=True,
    best_metric_key="median_center_error_mm",
    maximize_best_metric=False,
)

#### mIoU Comparison

In [ ]:
plot_multi_run(
    histories,
    labels,
    key="mean_iou",
    title="Mean IoU Comparison",
    ylabel="IoU",
    smooth=3,
    best_metric_key="median_center_error_mm",
    maximize_best_metric=False,
)

### Multiple Run Final Score Table 

In [ ]:
summary_rows = []

for label, hist in zip(labels, histories):
    row = {
        "run": label,
        "best_medCE": best_value(hist, "median_center_error_mm", maximize=False) if "median_center_error_mm" in list_metric_keys(hist) else float("nan"),
        "best_meanCE": best_value(hist, "mean_center_error_mm", maximize=False) if "mean_center_error_mm" in list_metric_keys(hist) else float("nan"),
        "best_P@T": best_value(hist, "p_at_thresh", maximize=True) if "p_at_thresh" in list_metric_keys(hist) else float("nan"),
        "best_mIoU": best_value(hist, "mean_iou", maximize=True) if "mean_iou" in list_metric_keys(hist) else float("nan"),
        "best_val_total_loss": best_value(hist, "val_total_loss", maximize=False) if "val_total_loss" in list_metric_keys(hist) else float("nan"),
        "best_val_heat_loss": best_value(hist, "val_heat_loss", maximize=False) if "val_heat_loss" in list_metric_keys(hist) else float("nan"),
        "best_val_size_loss": best_value(hist, "val_size_loss", maximize=False) if "val_size_loss" in list_metric_keys(hist) else float("nan"),
        "final_medCE": final_value(hist, "median_center_error_mm") if "median_center_error_mm" in list_metric_keys(hist) else float("nan"),
        "final_P@T": final_value(hist, "p_at_thresh") if "p_at_thresh" in list_metric_keys(hist) else float("nan"),
        "final_mIoU": final_value(hist, "mean_iou") if "mean_iou" in list_metric_keys(hist) else float("nan"),
    }
    summary_rows.append(row)

summary_rows

### Comparison of Choosen Metrics Set

In [ ]:
compare_keys = [
    "train_total_loss",
    "val_total_loss",
    "train_heat_loss",
    "val_heat_loss",
    "train_size_loss",
    "val_size_loss",
    "median_center_error_mm",
    "mean_center_error_mm",
    "p_at_thresh",
    "mean_iou",
]

for key in compare_keys:
    if all(key in list_metric_keys(h) for h in histories):
        plot_multi_run(
            histories,
            labels,
            key=key,
            title=f"Comparison: {key}",
            ylabel=key,
            smooth=3,
            as_percent=(key == "p_at_thresh"),
            best_metric_key="median_center_error_mm",
            maximize_best_metric=False,
        )

### Figure Save

In [ ]:
OUTDIR = Path("../outputs/analysis_plots")
OUTDIR.mkdir(parents=True, exist_ok=True)
print("Saving to:", OUTDIR)

In [ ]:
plot_multi_run(
    histories,
    labels,
    key="median_center_error_mm",
    title="Median Center Error Comparison",
    ylabel="mm",
    smooth=3,
)
save_current_figure(OUTDIR / "compare_median_center_error.png")